# innovfromtext: innovation and spillover indices from text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliuskoschnick/innovfromtext/blob/main/examples/innovfromtext_example.ipynb)

This notebook computes the three measures of [Koschnick (2025)](https://arxiv.org/abs/2512.16587) for science and technology in eighteenth-century England: the article titles of the *Philosophical Transactions* of the Royal Society, 1665–1800, and the descriptions of English patents, 1700–1800.

- **innovation**: is a text closer to its field's future than to its field's past?
- **spillover** A → B: is a text of field A closer to the future of field B than to B's past?
- **received spillover** A → B: is a text of field B closer to the recent past of field A than its closest peers in B were?

All three use the embeddings of [SteamBERTh](https://huggingface.co/juliuskoschnick/SteamBERTh), a historical BERT model fine-tuned on seventeenth- and eighteenth-century scientific and technical texts.

**On Colab, switch to a GPU first:** *Runtime → Change runtime type → T4 GPU*. On a CPU the notebook also runs, but embedding the texts takes several minutes instead of seconds.

In [1]:
try:
    import innovfromtext
except ImportError:
    %pip install -q git+https://github.com/juliuskoschnick/innovfromtext.git

In [2]:
import numpy as np
import pandas as pd
import innovfromtext as ift

pd.set_option("display.max_colwidth", 140)
print("innovfromtext", ift.__version__, "| device:", ift.resolve_device())

innovfromtext 0.1.0 | device: cuda


## 1. Data

The package needs one row per text with three columns: the **text**, the **publication year** (integer) and the **subject** (field). Here the fields are the *Philosophical Transactions* (science) and patents (technology). Patent descriptions are the short titles of Woodcroft (1854), with the words "improvement(s)" and "improved" removed as in the paper.

In [3]:
df = ift.load_example()
print(df.groupby("subject").year.agg(["size", "min", "max"]))
df.sample(4, random_state=1)

                            size   min   max
subject                                     
Patents                     2027  1700  1800
Philosophical Transactions  5554  1665  1800


,id,year,subject,text
7220,patent 2204,1797,Patents,making soap and bleaching by means and use of mineral and vegetable alkalis
6421,10.1098/rstl.1788.0011,1788,Philosophical Transactions,Experiments on the cooling of water below its freezing point
4388,10.1098/rstl.1764.0023,1764,Philosophical Transactions,"An account of the degree of cold observed in Bedfordshire: By John Howard, Esq; F. R. S. in a letter to John Canton, M. A. F. R. S"
1628,10.1098/rstl.1706.0023,1706,Philosophical Transactions,Of hydatides inclosed with a stony crust in the kidny of a sheep.


## 2. Embed once

`compute` embeds the texts itself if you give it no embeddings. Embedding is the slow step, so when you compute several measures it pays to embed once and pass the embeddings on. The first call downloads SteamBERTh (440 MB) from Hugging Face.

In [4]:
emb = ift.embed(df["text"])        # unit-length vectors, one row per text
emb.shape

(7581, 768)

## 3. Innovation index

$$\mathrm{Innovation}_{it_0} = \frac{f^k(v_{it_0}, \mathcal{F}_{t_0,\tau})}{f^k(v_{it_0}, \mathcal{B}_{t_0})}$$

where $f^k$ is the mean cosine similarity between text $i$ and the $k$ most similar texts of a pool, $\mathcal{F}_{t_0,\tau}$ holds the texts of the same field published in the $\tau$ years after $t_0$, and $\mathcal{B}_{t_0}$ all earlier texts of the field. Defaults: $\tau = k = 20$. With `return_similarities=True` the forward and backward similarities are returned as well.

Texts from the last $\tau$ years of the data get a missing index, because their forward window is incomplete (`complete_windows=True`). Texts from the first years of a field have only a few earlier texts to compare with, so their index is noisy. The patents in this example start in 1700, so we look at patents from 1720 onwards.

In [5]:
inn = ift.compute(df, "innovation", embeddings=emb, return_similarities=True)
inn.groupby("subject")[["innovation", "forward_similarity", "backward_similarity"]].describe().T.round(3)

subject                    Patents  Philosophical Transactions
innovation          count  874.000                    4729.000
                    mean     1.102                       0.969
                    std      0.385                       0.160
                    min      0.586                       0.399
                    25%      0.941                       0.891
                    50%      1.043                       0.959
                    75%      1.154                       1.025
                    max      7.555                       6.056
forward_similarity  count  874.000                    4729.000
                    mean     0.338                       0.483
                    std      0.067                       0.114
                    min      0.117                       0.172
                    25%      0.291                       0.401
                    50%      0.340                       0.487
                    75%      0.387                       0.565
                    max      0.523                       0.911
backward_similarity count  874.000                    4729.000
                    mean     0.322                       0.502
                    std      0.079                       0.110
                    min      0.029                       0.066
                    25%      0.275                       0.426
                    50%      0.328                       0.508
                    75%      0.375                       0.578
                    max      0.558                       0.925

In the list below, the top places among the articles go to the first years of the Lyndon weather registers, a format the *Transactions* then printed year after year: the index rewards texts that their field's future resembles.

In [6]:
# the most innovative patents and articles
keep = ((inn.subject == "Patents") & (inn.year >= 1720)) | ((inn.subject != "Patents") & (inn.year >= 1685))
(inn[keep].dropna(subset=["innovation"])
    .sort_values("innovation", ascending=False)
    .groupby("subject").head(4)[["year", "subject", "innovation", "text"]])

,year,subject,innovation,text
5065,1773,Philosophical Transactions,2.092396,"Extract of a register of the barometer, thermometer, and rain, at Lyndon in Rutland, 1772"
5140,1774,Philosophical Transactions,1.931259,"Extract of a register of the barometer, thermometer, and rain, at Lyndon in Rutland, 1773"
5292,1776,Philosophical Transactions,1.878121,"Meteorological Journal Kept at the House of The Royal Society, By Order of the President and Council"
1996,1720,Patents,1.805767,"heating, bending, and making of plank plyant of all sorts, to be wrought on any ship, vessel, lighter, boat, or floating craft, and for ..."
5196,1775,Philosophical Transactions,1.796702,"Extract of a register of the barometer, thermometer, and rain, at Lyndon, in Rutland, 1774"
2065,1722,Patents,1.731458,"machine for swimming or swimmers, to preserve people's lives in case of shipwreck at sea, the oversetting of boats, and in case of the c..."
3039,1744,Patents,1.722367,"specifick balsam, called the balsam of life"
2940,1742,Patents,1.714907,"a specifick powder which is a speedy & effectual cure for the rheumatism, & a perfect ease and relief if not a certain and effectual cur..."


## 4. Received spillover index

$$\text{Received spillover}_{A \to B}(v_{it}) = \frac{f^{k}(v_{it},\, \mathcal{B}^{A}_{t,\tau})}{f^{k}(\mathcal{P}^{k}_{it},\, \mathcal{B}^{A}_{t,\tau})}$$

A text of field B received a spillover from field A if it is closer to the recent past of A than its $k$ most similar predecessors in its own field (its peers $\mathcal{P}^k_{it}$) were. Here: did patents (B) draw on the recent *Philosophical Transactions* (A)? With several source fields, `received_spillover` is the mean over fields, and the index for each field is returned as well.

In [7]:
rec = ift.compute(df, "received_spillover", embeddings=emb,
                  from_fields=["Philosophical Transactions"], to_fields=["Patents"], return_similarities=True)
rec[rec.subject == "Patents"].filter(regex="spillover|similarity").describe().round(3)

,received_spillover,peer_similarity_from_philosophical_transactions,title_similarity_from_philosophical_transactions
count,2025.000,2025.000,2025.000
mean,0.590,0.335,0.198
std,0.109,0.043,0.048
min,0.238,0.222,0.072
25%,0.514,0.305,0.165
50%,0.587,0.334,0.192
75%,0.656,0.362,0.228
max,1.076,0.477,0.403


In [8]:
# mean log received spillover of patents by twenty-year period
rec["period"] = rec.year // 20 * 20
(rec[(rec.subject == "Patents") & (rec.year >= 1720)].dropna(subset=["received_spillover"])
    .assign(log_received_spillover=lambda d: np.log(d.received_spillover))
    .groupby("period").log_received_spillover.agg(["size", "mean"]).round(3))

,size,mean
period,,
1720,144,-0.594
1740,167,-0.567
1760,474,-0.566
1780,1093,-0.532
1800,91,-0.526


## 5. Spillover index

$$\mathrm{Spillover}_{i, A \to B} = \frac{f^k(v_{it_0}, \mathcal{F}^B_{t_0,\tau})}{f^k(v_{it_0}, \mathcal{B}^B_{t_0,\tau})}$$

A text of field A creates a spillover into field B if it is closer to B's next $\tau$ years than to B's previous $\tau$ years. Here: which articles of the *Philosophical Transactions* anticipated where patenting was heading? As the patents start in 1700, we look at articles from 1720 onwards. Very short titles give noisy ratios, so the list keeps titles of at least ten words.

In [9]:
spill = ift.compute(df, "spillover", embeddings=emb,
                    from_fields=["Philosophical Transactions"], to_fields=["Patents"])
spill["period"] = spill.year // 20 * 20
art = spill[(spill.subject != "Patents") & (spill.year >= 1720)].dropna(subset=["spillover"])
print(art.groupby("period").spillover.agg(["size", "median"]).round(3))
(art[art.text.str.split().str.len() >= 10]
    .sort_values("spillover", ascending=False)
    .head(5)[["year", "spillover", "text"]])

        size  median
period              
1720     636   1.116
1740    1196   1.211
1760     985   1.207
1780      49   1.175


,year,spillover,text
3267,1746,3.475293,"A continuation of a paper concerning electricity, by William Watson F. R. S. printed in these Trans. N. 477, Article I. ending p. 501"
3392,1750,2.834165,"On the agreement of barometers with the changes of air and weather: by Sam. Christiano Hollmann, Philos. in Acad. Professor of Gotiingen..."
2997,1743,2.633466,"Various medico-surgical observations by Johanne Daniele Schlichting, Med. & Chir. Doctor, Acad. Caesareo-Leopoldin. Nat. I am curious. M..."
4297,1763,2.621283,"The observation of the comet, which appeared in the month of May A. 1759, made by the Hague Com. à Peter Gabry, IVD Society Reg. Let me ..."
3011,1743,2.610386,"Observationes Cometæ, a R. P. Frantz Soc. Jes. factæ, mense Februario anni MDCCXLIII. Viennæ austriæ. Ex epistola Rev. Petri Isaaci Carn..."


## 6. Several measures at once, and your own data

`measure` also takes a list. For your own data, read a table with a text, a year and a subject column, name the columns, and save the result, e.g. as CSV for Stata or R. The indices are ratios; the paper uses their natural logarithm.

```python
mydata = pd.read_csv("my_texts.csv")              # columns e.g. abstract, pub_year, field
out = ift.compute(mydata, ["innovation", "received_spillover"],
                  text="abstract", year="pub_year", subject="field",
                  from_fields=["Physics"], tau=10, k=20, device="auto")
out.to_csv("my_texts_with_indices.csv", index=False)
```

Any sentence-transformers model can replace SteamBERTh, e.g. for modern texts: `model="sentence-transformers/all-mpnet-base-v2"`.

In [10]:
both = ift.compute(df, ["innovation", "received_spillover"], embeddings=emb,
                   from_fields=["Philosophical Transactions"], to_fields=["Patents"], show_progress=False)
both.columns.tolist()

['id', 'year', 'subject', 'text', 'innovation', 'received_spillover']